# Step 10–11 — Regime reconstruction & characterisation

This notebook takes the **final regime labels** (Step 9) and:

- **Step 10**: reconstructs regimes in **physical space** (Z500 anomaly composites)
- **Step 11**: characterises regimes statistically (**frequency, persistence, seasonality, transitions**)

**Inputs (from previous steps)**
- Step 9: `regime_labels(time)` (and optionally `centroids_pc`)
- Step 3: `z500_anom(time, lat, lon)`
- Step 4–6: `z500_norm(time, lat, lon)` and `z500_weighted(time, lat, lon)` (optional)

**Outputs (this notebook)**
- `regime_composite_anom(regime, lat, lon)`
- optional: `regime_composite_norm(regime, lat, lon)`, `regime_composite_weighted(regime, lat, lon)`
- tables/plots: frequency, persistence distribution, monthly occurrence (MJJAS), transition matrix

In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature

plt.rcParams.update({"figure.dpi": 120})


import os

from common.map_plotting_helpers import add_cartopy_land_ocean_basemap as _add_map_basemap
from common.preprocessing_helpers import (
    _is_netcdf_readable,
    bbox_step_filename_suffix,
    filter_by_sigma_mode as _filter_by_sigma_mode,
    filter_paths_by_yeo_johnson,
    find_module_root,
    pick_single as _pick_single,
    yeo_johnson_enabled,
)

module_root = find_module_root()

# ------------------------
# Configuration
# ------------------------

REGION = os.environ.get("REGION", "europe")  # "europe" | "northamerica"
BBOX_STEP: str | None = os.environ.get("BBOX_STEP", "B5")  # e.g. "B0".."B5"
BBOX_STEP = (str(BBOX_STEP).strip().upper() if BBOX_STEP else None)
SIGMA_MODE = str(os.environ.get("SIGMA_MODE", "")).strip().lower()

# Figure provenance tag
BBOX_TITLE = f"bbox={BBOX_STEP}" if BBOX_STEP else "bbox=full"
BBOX_FIGTAG = f"_bbox{BBOX_STEP}" if BBOX_STEP else ""

BBOX_OUT_TAG = bbox_step_filename_suffix(module_root, REGION, BBOX_STEP)

# Disambiguates Step 9/10-11 outputs from pre-YJ runs at the same k/w/eps/var/bbox
# (Step 7/8 already carry this via the `opt` label baked into their filenames).
YJ_TAG = "_yj" if yeo_johnson_enabled() else ""


def tag_figure_for_save(fig, *, prefix: str = "") -> None:
    """Add a bbox tag to the figure title (for saved figures)."""
    try:
        fig.suptitle(f"{prefix}{BBOX_TITLE}", y=1.02, fontsize=11)
    except Exception:
        pass


preprocessed_dir = module_root / "data" / "preprocessed" / REGION
results_clustering_dir = module_root / "results" / "clustering" / REGION
results_regimes_dir = module_root / "results" / "regimes" / REGION

YEAR_START = int(os.environ.get("YEAR_START", "1979"))
YEAR_END = int(os.environ.get("YEAR_END", "2025"))
YEAR_SPAN = os.environ.get("YEAR_SPAN", "") or (f"{YEAR_START}-{YEAR_END}" if YEAR_START != YEAR_END else f"{YEAR_START}")

# Keep consistent with Steps 3–9
WINDOW_DAYS = int(os.environ.get("WINDOW_DAYS", "11"))
EPSILON_M = float(os.environ.get("EPSILON_M", "20.0"))
VAR_THRESHOLD = float(os.environ.get("VAR_THRESHOLD", "0.90"))
FINAL_K = int(os.environ.get("FINAL_K", "7"))


# Same CLUSTER_MODE convention as notebook 09 ("ward" | "ward+kmeans"); the on-disk method tag
# ("ward" vs "kmeans") must match whatever notebook 09 actually wrote for this run.
def _normalize_cluster_mode(value: str) -> str:
    v = str(value).strip().lower().replace("-", "").replace("_", "")
    if v in ("ward", "w", "hrc", "wardonly"):
        return "ward"
    if v in ("ward+kmeans", "wardkmeans", "wardkm", "hybrid", "wk"):
        return "ward_kmeans"
    raise ValueError(f"CLUSTER_MODE must be 'ward' or 'ward+kmeans', got {value!r}")


CLUSTER_MODE = _normalize_cluster_mode(os.environ.get("CLUSTER_MODE", "ward"))
_METHOD_TAG = "ward" if CLUSTER_MODE == "ward" else "kmeans"
print(f"[step10_11] CLUSTER_MODE={CLUSTER_MODE!r} -> method tag {_METHOD_TAG!r}")


def resolve_step9_regimes(
    results_clustering_dir: Path,
    final_k: int,
    window_days: int,
    epsilon_m: float,
    var_threshold: float,
    *,
    year_span: str,
    bbox_step: str | None,
    method_tag: str,
) -> Path:
    forced = os.environ.get("IN_STEP9_REGIMES", "").strip()
    if forced:
        p = Path(forced).expanduser().resolve()
        if p.is_file() and _is_netcdf_readable(p):
            return p
    eps = int(round(epsilon_m))
    var = int(var_threshold * 100)
    pats = [
        f"09_Z500_regimes_{method_tag}_k{final_k:02d}_w{window_days:02d}_eps{eps:02d}_var{var:02d}_*{year_span}*.nc",
        f"09_Z500_regimes_{method_tag}_k{final_k:02d}_w{window_days:02d}_eps{eps:02d}_var{var:02d}_*_{year_span}.nc",
        f"09_Z500_regimes_{method_tag}_k{final_k:02d}_w{window_days:02d}_eps{eps:02d}_var{var:02d}_{year_span}*.nc",
        f"09_Z500_regimes_{method_tag}_k{final_k:02d}_w{window_days:02d}_eps{eps:02d}_var{var:02d}_{year_span}.nc",
        f"09_Z500_regimes_{method_tag}_k{final_k:02d}_w{window_days:02d}_eps{eps:02d}_var{var:02d}.nc",
    ]
    tok = (bbox_step or "").strip().upper()
    if tok:
        pats.extend(
            [
                f"09_Z500_regimes_{method_tag}_k{final_k:02d}_*step{tok}_*.nc",
                f"09_Z500_regimes_*_step{tok}_*.nc",
            ]
        )
    cands: list[Path] = []
    for pat in pats:
        cands.extend(sorted(results_clustering_dir.glob(pat)))
    cands = [p for p in cands if p.exists() and _is_netcdf_readable(p)]
    cands = filter_paths_by_yeo_johnson(cands)
    if tok:
        step_hits = [p for p in cands if f"_step{tok}_" in p.name]
        if step_hits:
            cands = step_hits
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in cands:
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return _pick_single(
        unique,
        kind=(
            f"Step 9 regimes (REGION={REGION}, method={method_tag}, k={final_k}, w={window_days}, "
            f"eps={eps}, var={var}, years={year_span}, bbox_step={bbox_step})"
        ),
    )


def resolve_step3_anomaly(preprocessed_dir: Path, window_days: int, *, year_span: str, bbox_step: str | None) -> Path:
    pats = [
        f"03_Z500_anomaly_w{window_days:02d}_*{year_span}*.nc",
        f"03_Z500_anomaly_w{window_days:02d}_*_{year_span}.nc",
        f"03_Z500_anomaly_w{window_days:02d}_{year_span}*.nc",
        f"03_Z500_anomaly_w{window_days:02d}_{year_span}.nc",
        f"03_Z500_anomaly_w{window_days:02d}.nc",
    ]

    cands: list[Path] = []
    for pat in pats:
        cands.extend(sorted(preprocessed_dir.glob(pat)))
    cands = [p for p in cands if p.exists() and _is_netcdf_readable(p)]

    if bbox_step:
        step = str(bbox_step).strip()
        step_hits = [p for p in cands if f"_step{step}_" in p.name]
        if step_hits:
            cands = step_hits
    cands = _filter_by_sigma_mode(cands, SIGMA_MODE, kind="Step 3 anomaly")
    cands = filter_paths_by_yeo_johnson(cands)

    # De-dup
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in cands:
        if p not in seen:
            seen.add(p)
            unique.append(p)

    if unique:
        return _pick_single(unique, kind=f"Step 3 anomaly (REGION={REGION}, w={window_days}, years={year_span}, bbox_step={bbox_step}, sigma_mode={SIGMA_MODE})")

    # Fallback: Step 2 pack carries z500_anom
    pack_pats = [
        f"02_z500_mjjas_climatology_anomalies_w{window_days:02d}_*_{year_span}.nc",
        f"02_z500_mjjas_climatology_anomalies_w{window_days:02d}_{year_span}.nc",
        f"02_z500_mjjas_climatology_anomalies_w{window_days:02d}.nc",
    ]
    pack_cands: list[Path] = []
    for pat in pack_pats:
        pack_cands.extend(sorted(preprocessed_dir.glob(pat)))
    pack_cands = [p for p in pack_cands if p.exists() and _is_netcdf_readable(p)]
    if bbox_step:
        step = str(bbox_step).strip()
        step_hits = [p for p in pack_cands if f"_step{step}_" in p.name]
        if step_hits:
            pack_cands = step_hits
    pack_cands = _filter_by_sigma_mode(pack_cands, SIGMA_MODE, kind="Step 2 anomaly pack")
    pack_cands = filter_paths_by_yeo_johnson(pack_cands)

    p = _pick_single(pack_cands, kind=f"Step 2 anomaly pack (fallback for Step 3 anomaly; REGION={REGION}, w={window_days}, years={year_span}, bbox_step={bbox_step}, sigma_mode={SIGMA_MODE})")
    warnings.warn(
        f"Step 3 file missing/unreadable; using Step 2 pack {p.name} (z500_anom).",
        UserWarning,
        stacklevel=2,
    )
    return p


def resolve_step4_6_preproc(
    preprocessed_dir: Path, window_days: int, epsilon_m: float, *, year_span: str, bbox_step: str | None
) -> Path:
    eps_label = int(round(float(epsilon_m)))
    pats = [
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_*{year_span}*.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_*_{year_span}.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_{year_span}*.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}_{year_span}.nc",
        f"04_05_06_Z500_preprocessed_w{window_days:02d}_eps{eps_label:02d}.nc",
    ]
    cands: list[Path] = []
    for pat in pats:
        cands.extend(sorted(preprocessed_dir.glob(pat)))
    cands = [p for p in cands if p.exists() and _is_netcdf_readable(p)]
    if bbox_step:
        step = str(bbox_step).strip()
        step_hits = [p for p in cands if f"_step{step}_" in p.name]
        if step_hits:
            cands = step_hits
    cands = _filter_by_sigma_mode(cands, SIGMA_MODE, kind="Step 4–6 preprocessed")
    cands = filter_paths_by_yeo_johnson(cands)
    # de-dup
    seen: set[Path] = set()
    unique: list[Path] = []
    for p in cands:
        if p not in seen:
            seen.add(p)
            unique.append(p)
    return _pick_single(unique, kind=f"Step 4–6 preprocessed (REGION={REGION}, w={window_days}, eps={eps_label}, years={year_span}, bbox_step={bbox_step}, sigma_mode={SIGMA_MODE})")


paths = {
    "in_regimes": resolve_step9_regimes(
        results_clustering_dir,
        FINAL_K,
        WINDOW_DAYS,
        EPSILON_M,
        VAR_THRESHOLD,
        year_span=YEAR_SPAN,
        bbox_step=BBOX_STEP,
        method_tag=_METHOD_TAG,
    ),
    "in_anom": resolve_step3_anomaly(preprocessed_dir, WINDOW_DAYS, year_span=YEAR_SPAN, bbox_step=BBOX_STEP),
    "in_norm": resolve_step4_6_preproc(preprocessed_dir, WINDOW_DAYS, EPSILON_M, year_span=YEAR_SPAN, bbox_step=BBOX_STEP),
    "out_step10_11": results_regimes_dir
    / f"10_11_Z500_regime_reconstruction_characterisation_{_METHOD_TAG}_k{FINAL_K:02d}_w{WINDOW_DAYS:02d}_eps{int(round(EPSILON_M)):02d}_var{int(VAR_THRESHOLD*100):02d}{YJ_TAG}_{YEAR_SPAN}{BBOX_OUT_TAG}.nc",
}

paths


In [ ]:
# Same figure builders as the CLI PDF (`summary_reports/step1011_render.py`).
from common.io_raw_composite import (
    pick_raw_composite_var,
    resolve_regime_composite_z500_raw_path,
)
from common.spatial import align_raw_composite_to_anomaly
from common.step1011_figures import (
    build_figure_composite_grid_notebook_style,
    build_figure_monthly_fraction,
    build_figure_persistence_histograms,
    build_figure_regime_anomaly_with_raw_contours,
    build_figure_transition_counts,
    build_figure_transition_probabilities,
)


In [ ]:
def _pick_name(candidates: tuple[str, ...], available: set[str], kind: str) -> str:
    for c in candidates:
        if c in available:
            return c
    raise KeyError(f"Could not infer {kind}. Tried {candidates}, available={sorted(list(available))}")


def infer_time_lat_lon(da: xr.DataArray) -> tuple[str, str, str]:
    time_dim = _pick_name(("time", "valid_time"), set(da.dims) | set(da.coords), "time dim")
    lat_dim = _pick_name(("lat", "latitude"), set(da.dims) | set(da.coords), "lat dim")
    lon_dim = _pick_name(("lon", "longitude"), set(da.dims) | set(da.coords), "lon dim")
    return time_dim, lat_dim, lon_dim


def run_length_table(labels: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Return (regime_ids, run_lengths) for consecutive runs in a 1D label series."""
    labels = np.asarray(labels).astype(int)
    if labels.size == 0:
        return np.array([], dtype=int), np.array([], dtype=int)

    regimes = [int(labels[0])]
    lengths = [1]
    for x in labels[1:]:
        x = int(x)
        if x == regimes[-1]:
            lengths[-1] += 1
        else:
            regimes.append(x)
            lengths.append(1)

    return np.asarray(regimes, dtype=int), np.asarray(lengths, dtype=int)


def transition_counts(labels: np.ndarray, regimes: np.ndarray) -> xr.DataArray:
    labels = np.asarray(labels).astype(int)
    regimes = np.asarray(regimes).astype(int)
    idx = {int(r): i for i, r in enumerate(regimes)}

    T = np.zeros((regimes.size, regimes.size), dtype=int)
    for a, b in zip(labels[:-1], labels[1:]):
        if int(a) in idx and int(b) in idx:
            T[idx[int(a)], idx[int(b)]] += 1

    return xr.DataArray(
        T,
        dims=("regime_from", "regime_to"),
        coords={"regime_from": regimes, "regime_to": regimes},
        name="transition_counts",
    )


def plot_transition_matrix(T: xr.DataArray, title: str, savepath=None) -> None:
    from pathlib import Path

    fig, ax = plt.subplots(figsize=(4.6, 4.0))
    arr = np.asarray(T.values)
    im = ax.imshow(arr, cmap="viridis")
    ax.set_title(title)
    ax.set_xlabel("to regime")
    ax.set_ylabel("from regime")
    ax.set_xticks(np.arange(arr.shape[1]))
    ax.set_yticks(np.arange(arr.shape[0]))
    ax.set_xticklabels(T["regime_to"].values)
    ax.set_yticklabels(T["regime_from"].values)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    for i in range(arr.shape[0]):
        for j in range(arr.shape[1]):
            ax.text(
                j,
                i,
                str(int(arr[i, j])),
                ha="center",
                va="center",
                color="white" if arr[i, j] > arr.max() / 2 else "black",
            )

    plt.tight_layout()
    if savepath is not None:
        savepath = Path(savepath)
        savepath.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(savepath, dpi=300, bbox_inches="tight")
    plt.show()

In [ ]:
# Load Step 9 regimes

ds_reg = xr.open_dataset(paths["in_regimes"], chunks=None)
if "regime_labels" not in ds_reg:
    raise KeyError(f"Expected 'regime_labels' in {paths['in_regimes']}, found: {list(ds_reg.data_vars)}")

labels_da = ds_reg["regime_labels"].load()
label_time_dim = labels_da.dims[0]
time_coord = labels_da[label_time_dim] if label_time_dim in labels_da.coords else ds_reg[label_time_dim]

regimes = np.unique(labels_da.values.astype(int))
regimes = regimes[np.isfinite(regimes)]
regimes = regimes[regimes > 0]
regimes = np.asarray(regimes, dtype=int)

print("Regimes:", regimes)
print("n_time:", labels_da.sizes[label_time_dim])

ds_reg

In [ ]:
# Load physical-space fields (anomalies, plus optional normalised/weighted)

ds_anom = xr.open_dataset(paths["in_anom"], chunks=None)
var_anom = "z500_anom" if "z500_anom" in ds_anom.data_vars else next(iter(ds_anom.data_vars))
z_anom = ds_anom[var_anom].load()

time_dim, lat_dim, lon_dim = infer_time_lat_lon(z_anom)

# Align times between labels and anomaly fields
z_anom = z_anom.rename({time_dim: label_time_dim}) if time_dim != label_time_dim else z_anom
z_anom = z_anom.sel({label_time_dim: time_coord})

# Optional: standardised and weighted fields from Step 4–6
have_norm = False
have_weighted = False
z_norm = None
z_weighted = None

try:
    ds_norm = xr.open_dataset(paths["in_norm"], chunks=None)
    if "z500_norm" in ds_norm.data_vars:
        z_norm = ds_norm["z500_norm"].load()
        t2, _, _ = infer_time_lat_lon(z_norm)
        z_norm = z_norm.rename({t2: label_time_dim}) if t2 != label_time_dim else z_norm
        z_norm = z_norm.sel({label_time_dim: time_coord})
        have_norm = True

    if "z500_weighted" in ds_norm.data_vars:
        z_weighted = ds_norm["z500_weighted"].load()
        t3, _, _ = infer_time_lat_lon(z_weighted)
        z_weighted = z_weighted.rename({t3: label_time_dim}) if t3 != label_time_dim else z_weighted
        z_weighted = z_weighted.sel({label_time_dim: time_coord})
        have_weighted = True
except FileNotFoundError:
    pass

print("z_anom:", z_anom.shape, z_anom.dims)
print("have_norm:", have_norm, "have_weighted:", have_weighted)
print("time coord equal:", int(z_anom.sizes[label_time_dim]) == int(labels_da.sizes[label_time_dim]))

z_anom

## Step 10 — Regime reconstruction (physical space)

Each regime composite is the mean Z500 anomaly field over all days assigned to that regime:

\[
\overline{Z}'_r(x,y) = \frac{1}{N_r}\sum_{t \in r} Z'(t,x,y)
\]

We compute composites for:
- **`z500_anom`** (recommended for physical interpretation)
- optionally **`z500_norm`** (standardised anomalies)
- optionally **`z500_weighted`** (area-weighted standardised anomalies; matches EOF/clustering input)

In [ ]:
def composite_by_regime(field: xr.DataArray, labels: xr.DataArray, regimes: np.ndarray) -> xr.DataArray:
    out = []
    for r in regimes:
        mask = labels == int(r)
        comp = field.where(mask).mean(dim=label_time_dim, skipna=True)
        out.append(comp)

    comp_all = xr.concat(out, dim=xr.IndexVariable("regime", regimes))
    comp_all.name = f"regime_composite_{field.name}" if field.name else "regime_composite"
    return comp_all


regime_composite_anom = composite_by_regime(z_anom, labels_da, regimes)

regime_composite_norm = None
regime_composite_weighted = None

if have_norm and z_norm is not None:
    regime_composite_norm = composite_by_regime(z_norm, labels_da, regimes)

if have_weighted and z_weighted is not None:
    regime_composite_weighted = composite_by_regime(z_weighted, labels_da, regimes)

regime_composite_anom

In [ ]:
# Regime composites: Cartopy basemap + raster + black contour overlay
n_reg = int(regimes.size)

# Force anomalies to geopotential meters (gpm) for plotting + colorbar label
units_in = str(regime_composite_anom.attrs.get("units", ""))
_units_l = units_in.lower()
_is_geopotential = ("m**2" in _units_l) or ("m2" in _units_l) or ("m^2" in _units_l) or ("s**-2" in _units_l) or ("s^-2" in _units_l)
regime_composite_anom_gpm = (regime_composite_anom / 9.80665) if _is_geopotential else regime_composite_anom
regime_composite_anom_gpm.attrs.update({"units": "gpm"})

vmax = float(np.nanpercentile(np.abs(regime_composite_anom_gpm.values), 98))

ncol = min(2, n_reg)
nrow = int(np.ceil(n_reg / ncol))

# Data are lon/lat in PlateCarree.
data_crs = ccrs.PlateCarree()

# Map extent from the full composite field
_lon = regime_composite_anom_gpm[lon_dim]
_lat = regime_composite_anom_gpm[lat_dim]
extent = [float(_lon.min()), float(_lon.max()), float(_lat.min()), float(_lat.max())]

cbar_label = "Geopotential Z500 anomalies (gpm)"
contour_levels = np.linspace(-vmax, vmax, 21)

fig2, axes2 = plt.subplots(
    nrow,
    ncol,
    # Cartopy GeoAxes often look too tall; reduce per-row height to avoid large row gaps
    figsize=(3.4 * ncol, 2.3 * nrow),
    constrained_layout=False,
    subplot_kw={"projection": data_crs},
)
axes2 = np.atleast_1d(axes2).ravel()

# Tighten whitespace between panels
fig2.subplots_adjust(wspace=0.02, hspace=0.01, top=0.93, bottom=0.08)

im2 = None
for i, r in enumerate(regimes):
    ax = axes2[i]
    m = regime_composite_anom_gpm.sel(regime=int(r))
    im2 = ax.pcolormesh(
        m[lon_dim],
        m[lat_dim],
        m,
        shading="auto",
        cmap="RdBu_r",
        vmin=-vmax,
        vmax=vmax,
        alpha=0.6,
        transform=data_crs,
        zorder=2,
    )
    _add_map_basemap(ax, extent=extent, zorder=0)
    ax.contour(
        m[lon_dim],
        m[lat_dim],
        m.values,
        levels=contour_levels,
        colors="k",
        linewidths=0.75,
        transform=data_crs,
        zorder=5,
    )
    ax.set_title(f"Regime {int(r)}")

    row, col = divmod(i, ncol)
    is_bottom = row == nrow - 1
    is_left = col == 0
    ax.tick_params(
        axis="x",
        which="both",
        bottom=is_bottom,
        top=False,
        labelbottom=is_bottom,
    )
    ax.tick_params(
        axis="y",
        which="both",
        left=is_left,
        right=False,
        labelleft=is_left,
    )
    if is_bottom:
        ax.set_xlabel(lon_dim)
    if is_left:
        ax.set_ylabel(lat_dim)

for j in range(n_reg, axes2.size):
    axes2[j].axis("off")

fig2.colorbar(im2, ax=axes2[:n_reg].tolist(), fraction=0.046, pad=0.02, label=cbar_label)

from pathlib import Path
Path("figures").mkdir(parents=True, exist_ok=True)
tag_figure_for_save(fig2, prefix=f"{REGION.upper()} | ")
fig2.savefig(Path("figures") / f"EU_step10_11_regime_composites_anom_raster_contours{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

plt.show()

print("Color scale: ±", vmax)


In [ ]:
# 2c) Plot regime composites of **raw Z500** only (not anomalies)
#
# Preferred workflow (fast): compute raw composites on Levante (small output NetCDF), rsync back, then plot.
# Fallback workflow (slow / large I/O): if the composite file is missing locally, compute composites on-the-fly
# from the local raw ERA5 Z500 NetCDF in `data/raw/era5/z500/<REGION>/...`.

import inspect
from pathlib import Path

if "data_crs" not in globals():
    data_crs = ccrs.PlateCarree()

from common.preprocessing_helpers import find_module_root

module_root = find_module_root() if "module_root" not in globals() else module_root
REGION = "europe" if "REGION" not in globals() else REGION
YEAR_SPAN = YEAR_SPAN if "YEAR_SPAN" in globals() else "1979-2025"
FINAL_K = FINAL_K if "FINAL_K" in globals() else 7

# ------------------------------
# 1) Try to load precomputed raw-composite file (small)
# ------------------------------
results_regimes_dir = module_root / "results" / "regimes" / REGION

p_comp = results_regimes_dir / f"10_regime_composite_z500_raw_k{int(FINAL_K):02d}_{YEAR_SPAN}.nc"
if not p_comp.exists():
    p_comp = results_regimes_dir / f"10_regime_composite_z500_raw_{YEAR_SPAN}.nc"

regime_composite = None
n_days_da = None
comp_var = None

if p_comp.exists():
    print("Loading precomputed raw Z500 composites:", p_comp)
    ds_comp = xr.open_dataset(p_comp, chunks=None)

    comp_candidates = [v for v in ds_comp.data_vars if v.startswith("regime_composite_")]
    if not comp_candidates:
        raise KeyError(f"No regime_composite_* variable found in {p_comp}. data_vars={list(ds_comp.data_vars)}")

    comp_var = "regime_composite_z500_raw" if "regime_composite_z500_raw" in comp_candidates else comp_candidates[0]
    regime_composite = ds_comp[comp_var]

    if "n_days" in ds_comp:
        n_days_da = ds_comp["n_days"]

# ------------------------------
# 2) Fallback: compute composites from local raw ERA5 Z500 file (large)
# ------------------------------
if regime_composite is None:
    if "labels_da" not in globals() or "regimes" not in globals():
        raise RuntimeError("This cell expects Step 9 labels already loaded as `labels_da` and `regimes`. Run the earlier 'Load Step 9 regimes' cell first.")

    raw_dir = module_root / "data" / "raw" / "era5" / "z500" / REGION

    # You can override the path by defining RAW_Z500_FILE in the notebook before running this cell.
    if "RAW_Z500_FILE" in globals() and RAW_Z500_FILE is not None:
        p_raw = Path(RAW_Z500_FILE)
    else:
        hits = sorted(raw_dir.glob("era5_z500_*.nc"))
        if not hits:
            hits = sorted(raw_dir.glob("*.nc"))
        if not hits:
            raise FileNotFoundError(f"No raw ERA5 Z500 NetCDF found under {raw_dir}.")
        p_raw = hits[-1]

    print("Computing raw Z500 composites from:", p_raw)
    ds_raw = xr.open_dataset(p_raw, chunks=None)

    # Pick variable (common names: 'z', 'z500', 'geopotential')
    var_candidates = [v for v in ("z500", "z", "geopotential") if v in ds_raw.data_vars]
    z_var = var_candidates[0] if var_candidates else list(ds_raw.data_vars)[0]
    z_raw = ds_raw[z_var]

    # Align time coordinate with labels
    label_time_dim = labels_da.dims[0]
    raw_time_dim = "time" if "time" in z_raw.dims else ("valid_time" if "valid_time" in z_raw.dims else z_raw.dims[0])

    if raw_time_dim != label_time_dim:
        z_raw = z_raw.rename({raw_time_dim: label_time_dim})

    # Many raw ERA5 files are at 12:00 UTC while labels are daily at 00:00.
    # Normalize both to calendar date so we can align by day.
    import pandas as pd

    z_time = pd.to_datetime(z_raw[label_time_dim].values).normalize()
    l_time = pd.to_datetime(labels_da[label_time_dim].values).normalize()

    z_raw = z_raw.assign_coords({label_time_dim: (label_time_dim, z_time)})
    labels_norm = labels_da.assign_coords({label_time_dim: (label_time_dim, l_time)})

    # Inner-join alignment (keeps only overlapping days)
    z_raw_aligned, labels_aligned = xr.align(z_raw, labels_norm, join="inner")

    if z_raw_aligned.sizes[label_time_dim] == 0:
        raise ValueError(
            f"No overlapping dates between raw Z500 ({p_raw}) and labels ({paths.get('in_regimes', 'Step9 file')}). "
            "Tip: check the raw file time stamps (00 vs 12 UTC) and that both cover the same MJJAS season and years."
        )

    # Compute composites using the helper defined earlier in the notebook
    regime_composite = composite_by_regime(z_raw_aligned, labels_aligned, regimes)
    comp_var = f"regime_composite_{z_var}_raw"
    regime_composite = regime_composite.rename(comp_var)

    # Days per regime
    n_days_da = xr.concat([(labels_aligned == int(r)).sum(dim=label_time_dim) for r in regimes], dim="regime")
    n_days_da = n_days_da.assign_coords(regime=("regime", regimes)).astype(int)
    n_days_da.name = "n_days"


In [ ]:
# 2d) Combined plot: filled **Z500 anomalies (gpm)** + contourlines of **mean raw Z500 (m)**
# Same builder as the CLI PDF (`regime_plotting` / `common.step1011_render`).

from pathlib import Path

import matplotlib.pyplot as plt

if "regime_composite_anom" not in globals():
    raise NameError("regime_composite_anom is not defined. Run the 2b anomaly-composites cell first.")
if "regime_composite" not in globals():
    raise NameError("regime_composite is not defined. Run the 2c raw-composites cell first.")

anom0 = regime_composite_anom
raw0 = regime_composite
raw_on_grid, _used_interp = align_raw_composite_to_anomaly(anom0, raw0)
if raw_on_grid is None:
    raise ValueError("Could not align raw composite to anomaly grid for 2d plot.")

_bbox = str(BBOX_STEP).strip().upper() if BBOX_STEP else "B0"
_k = int(FINAL_K)

fig = build_figure_regime_anomaly_with_raw_contours(
    anom0,
    raw_on_grid,
    bbox_step=_bbox,
    k=_k,
    fig_num=None,
    footer_line=None,
)

Path("figures").mkdir(parents=True, exist_ok=True)
fig.savefig(
    Path("figures") / f"EU_step10_11_regime_anom_with_raw_contours{BBOX_FIGTAG}.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()



In [ ]:

# ------------------------------
# Plotting (each panel: **mean raw Z500** in original file units, e.g. ERA5 geopotential m²/s²)
# ------------------------------


# --- Consistent plot styling as in cells 1-109 (standard look and feel) ---

# Infer lon/lat names used in this file
lon_dim = 'longitude' if 'longitude' in regime_composite.coords else 'lon'
lat_dim = 'latitude' if 'latitude' in regime_composite.coords else 'lat'

# Basic sanity check: fail with useful message on empty spatial dims
if regime_composite.sizes.get(lon_dim, 0) == 0 or regime_composite.sizes.get(lat_dim, 0) == 0:
    raise ValueError(f"Empty spatial dims: {lon_dim}={regime_composite.sizes.get(lon_dim)}, {lat_dim}={regime_composite.sizes.get(lat_dim)}")

# Always ensure full realization in memory for plotting (avoid lazy dask issues)
regime_composite = regime_composite.astype(np.float64).load()

vals = regime_composite.values
if not np.isfinite(vals).any():
    raise ValueError(
        "regime_composite is all-NaN / non-finite, so there is nothing to plot. "
        "This usually means the composite file was created with a different domain, time window, or variable name."
    )

# Normalize longitude to [-180, 180] for cleaner global maps
lon = regime_composite[lon_dim]
try:
    lon_max = float(lon.max())
except Exception:
    lon_max = None

if lon_max is not None and lon_max > 180:
    lon_wrapped = ((lon + 180) % 360) - 180
    regime_composite = regime_composite.assign_coords({lon_dim: lon_wrapped}).sortby(lon_dim)

# ERA5-style grids may be N->S; always sort south->north for pcolormesh
lat_1d = regime_composite[lat_dim]
if lat_1d.size > 1 and float(lat_1d[0]) > float(lat_1d[-1]):
    regime_composite = regime_composite.sortby(lat_dim)

vals = regime_composite.values  # Get array again in case it changed

extent = [
    float(regime_composite[lon_dim].min()),
    float(regime_composite[lon_dim].max()),
    float(regime_composite[lat_dim].min()),
    float(regime_composite[lat_dim].max()),
]

# Lon/lat = PlateCarree transformation
map_transform = ccrs.PlateCarree()

# Layout and style: min/max, colormap, axes layout
_regimes = regime_composite['regime'].values.astype(int)
regimes_plot = _regimes
n_reg = int(len(regimes_plot))

ncol = min(2, n_reg)
nrow = int(np.ceil(n_reg / ncol))

# colormap: robust percentiles, vmin/vmax logic
vmin, vmax = (float(x) for x in np.nanpercentile(vals, [2, 98]))
if not (np.isfinite(vmin) and np.isfinite(vmax) and vmax > vmin):
    vmin, vmax = float(np.nanmin(vals)), float(np.nanmax(vals))
if not (np.isfinite(vmin) and np.isfinite(vmax) and vmax > vmin):
    vmin, vmax = 0.0, 1.0

# Styling: red–blue colormap for Z500 (same as regime anomaly maps)
cmap_c = plt.get_cmap("RdBu_r")
alpha_c = 0.6
levels_c = np.linspace(vmin, vmax, 8)

# Colorbar label
units = getattr(regime_composite, "attrs", {}).get("units")
cbar_label = f"{regime_composite.name} ({units})" if units else (regime_composite.name or "value")

# Figure set-up
fig2c, axes2c = plt.subplots(
    nrow,
    ncol,
    figsize=(3.5 * ncol, 2.5 * nrow),
    constrained_layout=False,
    subplot_kw={'projection': data_crs},
)
axes2c = np.atleast_1d(axes2c).ravel()
fig2c.subplots_adjust(wspace=0.035, hspace=0.08, left=0.04, right=0.93, top=0.89, bottom=0.10)

im2c = None
for i, r in enumerate(regimes_plot):
    ax = axes2c[i]
    r_int = int(r)
    m = regime_composite.sel(regime=r_int)
    m2 = m.squeeze(drop=True)
    if {lat_dim, lon_dim}.issubset(m2.dims):
        m2 = m2.transpose(lat_dim, lon_dim)

    n_days = None
    if n_days_da is not None and "regime" in n_days_da.dims:
        n_days = int(n_days_da.sel(regime=r_int).values)

    # Draw filled field first
    im2c = ax.pcolormesh(
        m2[lon_dim],
        m2[lat_dim],
        m2,
        shading="auto",
        cmap=cmap_c,
        vmin=vmin,
        vmax=vmax,
        alpha=alpha_c,
        transform=map_transform,
        zorder=2,
        rasterized=True,
    )
    _add_map_basemap(
        ax,
        extent=extent,
        zorder=0,
        linewidth=0.7,
        ocean_color="#e5ecf6",  # light blue for style consistency
        coastline_color="0.15",
    )

    # Contours overlay
    levels_panel = levels_c
    if levels_panel is None or len(levels_panel) < 5:
        # fallback: autoscale
        try:
            mn, mx = float(np.nanmin(m2.values)), float(np.nanmax(m2.values))
        except Exception:
            mn, mx = vmin, vmax
        levels_panel = np.linspace(mn, mx, 8)
    cs = ax.contour(
        m2[lon_dim], m2[lat_dim], m2.values,
        levels=levels_panel,
        colors="k",
        linewidths=0.85,
        alpha=0.82,
        transform=map_transform,
        zorder=5,
    )
    ax.clabel(cs, inline=True, fontsize=7, fmt="%.0f", rightside_up=True, colors="k")

    # Panel title: regime number and count
    title = f"Regime {r_int}" if n_days is None else f"Regime {r_int} (N={n_days})"
    ax.set_title(title, fontsize=11, fontweight="normal", pad=4)

    # Clean ticks: only edge-most
    row, col = divmod(i, ncol)
    is_bottom = row == nrow - 1
    is_left = col == 0
    ax.tick_params(axis="x", which="both", bottom=is_bottom, top=False, labelbottom=is_bottom, labelsize=8)
    ax.tick_params(axis="y", which="both", left=is_left, right=False, labelleft=is_left, labelsize=8)
    if is_bottom:
        ax.set_xlabel(lon_dim, fontsize=9)
    if is_left:
        ax.set_ylabel(lat_dim, fontsize=9)

# Turn off empty axes panels
for j in range(n_reg, axes2c.size):
    axes2c[j].axis("off")

# Colorbar: slightly thinner for consistent style
cbar = fig2c.colorbar(
    im2c,
    ax=axes2c[:n_reg].tolist(),
    fraction=0.047,
    pad=0.02,
    aspect=25,
    label=cbar_label,
    location='right'
)
cbar.ax.tick_params(labelsize=8)
cbar.set_label(cbar_label, fontsize=10)

from pathlib import Path
outdir = Path("figures")
outdir.mkdir(parents=True, exist_ok=True)
_safe = "".join(ch if (ch.isalnum() or ch in "-_.") else "_" for ch in str(comp_var))
tag_figure_for_save(fig2c, prefix=f"{REGION.upper()} | ")
fig2c.savefig(outdir / f"EU_step10_11_regime_composite_{_safe}{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

plt.show()

print("Plot extent:", extent)
print("Data finite fraction:", float(np.isfinite(vals).mean()))
print("Color scale (mean raw Z500, 2–98 pct):", vmin, "to", vmax)
print("Plotted variable:", comp_var)


### Interpretation guide (how to read composites)

Each composite map represents a **canonical circulation anomaly pattern** for that regime:

- **Ridges/troughs**: persistent positive/negative Z500 anomalies
- **Large-scale wave structure**: regime-scale (not synoptic noise)
- **Flow configuration**: downstream impacts on storm tracks and moisture transport

A good composite should show **coherent, large-scale structure** rather than small-scale speckle.

## Step 11 — Regime characterisation

We quantify:

- **Frequency**: occurrence probability per regime
- **Persistence**: distribution of consecutive-day run lengths per regime
- **Seasonality (MJJAS)**: monthly regime frequency (May–Sep)
- **Transitions**: preferred regime-to-regime changes (optional but recommended)

In [ ]:
labels = labels_da.values.astype(int)

# 11.1 Frequency
counts = np.array([(labels == int(r)).sum() for r in regimes], dtype=int)
frequency = counts / counts.sum()

freq_da = xr.DataArray(
    frequency.astype(np.float32),
    dims=("regime",),
    coords={"regime": regimes},
    name="frequency",
)

freq_da.to_pandas().rename("frequency").to_frame().assign(percent=lambda df: 100 * df["frequency"])

In [ ]:
# 11.2 Persistence (run-length distributions)

run_regime, run_len = run_length_table(labels)

runs = xr.Dataset(
    {
        "run_regime": xr.DataArray(run_regime.astype(np.int16), dims=("episode",)),
        "run_length": xr.DataArray(run_len.astype(np.int16), dims=("episode",)),
    }
)

pers_summary = []
for r in regimes:
    L = run_len[run_regime == int(r)]
    pers_summary.append(
        {
            "regime": int(r),
            "n_episodes": int(L.size),
            "mean_days": float(np.mean(L)) if L.size else float("nan"),
            "median_days": float(np.median(L)) if L.size else float("nan"),
            "p90_days": float(np.percentile(L, 90)) if L.size else float("nan"),
        }
    )

pers_df = (
    __import__("pandas")
    .DataFrame(pers_summary)
    .set_index("regime")
    .sort_index()
)

pers_df

In [ ]:
# Persistence distribution plots

import pandas as pd

max_len = int(np.max(run_len)) if run_len.size else 1

ncol = min(2, int(regimes.size))
nrow = int(np.ceil(regimes.size / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=(4.0 * ncol, 2.8 * nrow), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for i, r in enumerate(regimes):
    ax = axes[i]
    L = run_len[run_regime == int(r)]
    ax.hist(L, bins=np.arange(1, max_len + 2) - 0.5, alpha=0.85)
    ax.set_title(f"Regime {int(r)} persistence")
    ax.set_xlabel("episode length (days)")
    ax.set_ylabel("count")
    ax.grid(True, axis="y", alpha=0.25)

for j in range(regimes.size, axes.size):
    axes[j].axis("off")

from pathlib import Path
Path("figures").mkdir(parents=True, exist_ok=True)
tag_figure_for_save(fig, prefix=f"{REGION.upper()} | ")
fig.savefig(Path("figures") / f"EU_step10_11_persistence_histograms{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# 11.3 Seasonality (monthly distribution within MJJAS)

time_index = pd.to_datetime(time_coord.values)
months = pd.Index(time_index.month, name="month")

month_regime = pd.DataFrame({"month": months.values, "regime": labels})
month_counts = (
    month_regime.groupby(["month", "regime"]).size().unstack("regime").reindex(index=[5, 6, 7, 8, 9])
)
month_frac = month_counts.div(month_counts.sum(axis=1), axis=0)

month_frac

In [ ]:
# Plot monthly regime fractions

fig, ax = plt.subplots(figsize=(7.0, 3.2))
month_frac.plot(kind="bar", stacked=True, ax=ax, colormap="tab20")
ax.set_title("Regime occurrence by month (fractions)")
ax.set_xlabel("month")
ax.set_ylabel("fraction of days")
ax.grid(True, axis="y", alpha=0.25)
ax.legend(title="regime", bbox_to_anchor=(1.02, 1.0), loc="upper left")
plt.tight_layout()

from pathlib import Path
Path("figures").mkdir(parents=True, exist_ok=True)
tag_figure_for_save(fig, prefix=f"{REGION.upper()} | ")
fig.savefig(Path("figures") / f"EU_step10_11_monthly_regime_occurrence_fractions{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

plt.show()

In [ ]:
# 11.4 Transition behaviour (counts + probabilities)

T_counts = transition_counts(labels, regimes)

from pathlib import Path
Path("figures").mkdir(parents=True, exist_ok=True)
plot_transition_matrix(
    T_counts,
    title="Transition counts",
    savepath=Path("figures") / "EU_step10_11_transition_counts.png",
)

row_sum = T_counts.sum("regime_to")
T_prob = xr.where(row_sum > 0, T_counts / row_sum, np.nan).rename("transition_prob")

fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(T_prob.values, cmap="magma", vmin=0, vmax=float(np.nanmax(T_prob.values)))
ax.set_title("Transition probabilities (row-normalised)")
ax.set_xlabel("to regime")
ax.set_ylabel("from regime")
ax.set_xticks(np.arange(regimes.size))
ax.set_yticks(np.arange(regimes.size))
ax.set_xticklabels(regimes)
ax.set_yticklabels(regimes)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()

from pathlib import Path
Path("figures").mkdir(parents=True, exist_ok=True)
tag_figure_for_save(fig, prefix=f"{REGION.upper()} | ")
fig.savefig(Path("figures") / f"EU_step10_11_transition_probabilities{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

plt.show()

T_prob

## Validation checks (recommended)

- **Physical consistency**: composites should show coherent large-scale wave patterns (not noisy speckle).
- **Balance of regimes**: no single regime should dominate (rule-of-thumb: >60% is suspicious) and very rare regimes (<5%) should be justified.
- **Temporal coherence**: regimes should persist for multiple days; extreme day-to-day switching suggests clustering instability.
- **Seasonal realism (MJJAS)**: expect differences between early/late summer; transitional months can show mixed behaviour.

In [ ]:
# Save Step 10–11 outputs to NetCDF (lightweight, analysis-ready)

out = xr.Dataset(attrs=dict(ds_reg.attrs))

out["regime_labels"] = labels_da
out["frequency"] = freq_da
out["regime_composite_anom"] = regime_composite_anom.astype("float32")

# Embed raw Z500 composite on the anomaly grid so PDF summaries match notebook 2d without a separate NetCDF.
_rc = globals().get("regime_composite")
if _rc is not None:
    try:
        from common.spatial import align_raw_composite_to_anomaly

        _raw_on, _used_interp = align_raw_composite_to_anomaly(regime_composite_anom, _rc)
        if _raw_on is not None:
            out["regime_composite_z500_raw"] = _raw_on.astype("float32")
    except Exception as _e:
        print("Could not add regime_composite_z500_raw to Step 10-11 NetCDF:", _e)

if regime_composite_norm is not None:
    out["regime_composite_norm"] = regime_composite_norm.astype("float32")

if regime_composite_weighted is not None:
    out["regime_composite_weighted"] = regime_composite_weighted.astype("float32")

out["transition_counts"] = T_counts.astype("int32")
out["transition_prob"] = T_prob.astype("float32")

out["run_regime"] = runs["run_regime"]
out["run_length"] = runs["run_length"]

# Monthly fractions (store May–Sep rows; months as coordinate)
month_frac_xr = xr.DataArray(
    month_frac.values.astype(np.float32),
    dims=("month", "regime"),
    coords={"month": month_frac.index.values.astype(int), "regime": month_frac.columns.values.astype(int)},
    name="monthly_fraction",
)
out["monthly_fraction"] = month_frac_xr

out.attrs.update(
    {
        "step": "10_11_regime_reconstruction_characterisation",
        "source_regimes": str(paths["in_regimes"]),
        "source_anom": str(paths["in_anom"]),
        "source_norm": str(paths["in_norm"]),
        "note": "Composites computed by averaging fields over days assigned to each regime label.",
    }
)

paths["out_step10_11"].parent.mkdir(parents=True, exist_ok=True)
out.to_netcdf(paths["out_step10_11"])

paths["out_step10_11"]

## Evaluation — Do Aug–Sep regimes co-vary with European blocking?

We use the blocking diagnostics in `Blocking/data/Blocking-output-extented/*/3Blockingproperties_jederZeitschritt_hol+Omega_python.csv`.

For each year, we compute:
- Aug–Sep **regime occurrence**: number of days assigned to each regime
- Aug–Sep **blocking occurrence**: number of days with at least one **HoL** block and/or **Omega** block

Then we compute (year-to-year) **Pearson correlations** between each regime’s occurrence and the blocking occurrence metrics.

In [ ]:
# --- Germany polygon for trapezoid intersection (VG5000) ---

# This cell builds `germany_union` in lon/lat (EPSG:4326), for intersection
# against the blocking trapezoids derived from NCEP lon/lat.

try:
    import geopandas as gpd
except ImportError:
    gpd = None
    print("geopandas not available; Germany trapezoid filter will be disabled.")

# This cell runs before the later shapely import cell, so avoid relying on a
# previously defined `Polygon` symbol when the notebook is executed from scratch.
if gpd is None:
    germany_union = None
else:
    shp_rel = Path(
        "meta_data/shapefiles/vg5000_01-01.utm32s.shape.ebenen/vg5000_ebenen_0101/VG5000_LAN.shp"
    )
    shp_path = (module_root.parent / shp_rel).resolve()

    if not shp_path.exists():
        print(f"Germany shapefile not found at {shp_path}")
        germany_union = None
    else:
        germany_admin = gpd.read_file(shp_path)

        # VG5000 is typically UTM32N (EPSG:25832). Regardless, we convert to lon/lat
        # because the blocking trapezoids are built from lon/lat coordinates.
        germany_admin = germany_admin.to_crs("EPSG:4326")

        # Keep only Germany polygons (some admin layers can include multiple features)
        geoms = [x for x in germany_admin.geometry if x is not None and not x.is_empty]
        if not geoms:
            print("Germany shapefile contained no geometries; filter disabled.")
            germany_union = None
        else:
            try:
                from shapely import unary_union as _uu
            except ImportError:
                from shapely.ops import unary_union as _uu

            germany_union = _uu(geoms)
            print(f"Loaded Germany polygon from {shp_path.name} (EPSG:4326).")


In [ ]:
# --- Blocking vs regimes (Aug–Sep) ---
from __future__ import annotations

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xarray as xr
import os
import pickle
from functools import lru_cache

try:
    import geopandas as _gpd

    _HAS_GPD = True
except ImportError:
    _gpd = None  # type: ignore
    _HAS_GPD = False

try:
    from shapely.geometry import Polygon
    from shapely.prepared import prep as _shapely_prep
except ImportError:
    Polygon = None  # type: ignore
    _shapely_prep = None

# ------------------------
# Config
# ------------------------
MONTHS = (8, 9)  # Aug, Sep
blocking_glob = "*/3Blockingproperties_jederZeitschritt_hol+Omega_python.csv"

# Notebook is in Paper3_canada/1_Detect_Weather_Patterns/src/.
# Blocking folder is Paper3_canada/Blocking/.
#
# Resolution order:
#   1. Explicit override: env var BLOCKING_DATA_ROOT or a Python global of the
#      same name (point at the directory that contains per-year subfolders, i.e.
#      the equivalent of `<repo>/Blocking/data/Blocking-output-extented`).
#   2. Conventional repo layout relative to module_root / Path.cwd().
#   3. Last-resort walk-up.
# If nothing resolves, this cell SKIPS gracefully — the blocking diagnostic is
# optional on systems where the Blocking dataset isn't synced (e.g. TU HPC).
_here = Path.cwd().resolve()
_candidates: list[Path] = []

_override = globals().get("BLOCKING_DATA_ROOT", None) or os.environ.get("BLOCKING_DATA_ROOT", "").strip()
if _override:
    _candidates.append(Path(_override).expanduser().resolve())

_candidates += [
    (_here.parent / "Blocking" / "data" / "Blocking-output-extented"),  # if cwd=.../Paper3_canada/1_Detect_Weather_Patterns
    (_here.parent.parent / "Blocking" / "data" / "Blocking-output-extented"),  # if cwd=.../Paper3_canada/1_Detect_Weather_Patterns/src
]

# If earlier cells defined module_root, use it as well
if "module_root" in globals():
    _c = Path(module_root).resolve().parent / "Blocking" / "data" / "Blocking-output-extented"
    _candidates.insert(1 if _override else 0, _c)

blocking_root = None
for c in _candidates:
    if c.exists():
        blocking_root = c
        break

# Last resort: walk up a few levels and look for the expected folder
if blocking_root is None:
    for p in [_here, *_here.parents][:6]:
        c = p / "Blocking" / "data" / "Blocking-output-extented"
        if c.exists():
            blocking_root = c
            break

if blocking_root is None:
    print(
        "[blocking diagnostic] SKIP: could not locate "
        "'Blocking/data/Blocking-output-extented'.\n"
        "  Tried:\n    - "
        + "\n    - ".join(str(x) for x in _candidates)
        + "\n  Set BLOCKING_DATA_ROOT (env var or notebook global) to that "
        "directory to enable this section, or ignore this message if the "
        "blocking dataset is not available on this machine."
    )
else:
    print(f"[blocking diagnostic] using blocking_root={blocking_root}")

# Optional: path to a Germany polygon shapefile (any CRS; reprojected to EPSG:4326).
# If unset, falls back to Natural Earth country boundaries via geopandas (if installed).
GERMANY_SHP = globals().get("GERMANY_SHP", None)
if GERMANY_SHP is not None:
    GERMANY_SHP = Path(GERMANY_SHP)
_env_shp = os.environ.get("BLOCKING_GERMANY_SHP", "").strip()
if GERMANY_SHP is None and _env_shp:
    GERMANY_SHP = Path(_env_shp)

# ------------------------
# Helpers
# ------------------------

def _read_blocking_csv(path: Path) -> pd.DataFrame:
    """Read blocking per-timestep file.

    File has one comment line + one header line starting with '#'.
    Data rows may be wrapped in quotes; use csv.reader for robust parsing.
    """
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        _ = f.readline()  # comment
        header_line = f.readline()

    if not header_line:
        raise ValueError(f"Missing header in {path}")

    header = [h.strip() for h in header_line.lstrip("#").strip().split(",")]
    if "Zeit" not in header:
        raise KeyError(f"Expected a 'Zeit' column in {path} header")
    if "HoL5" not in header:
        raise KeyError(f"Expected a 'HoL5' column in {path} header")
    if "Blocknummer" not in header:
        raise KeyError(f"Expected a 'Blocknummer' column in {path} header")
    if "Zeitschritt" not in header:
        raise KeyError(f"Expected a 'Zeitschritt' column in {path} header")

    idx_block = header.index("Blocknummer")
    idx_hol5 = header.index("HoL5")
    idx_zeit = header.index("Zeit")
    idx_zs = header.index("Zeitschritt")

    rows = []
    with path.open("r", encoding="utf-8", errors="replace", newline="") as f:
        next(f, None)
        next(f, None)
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if len(line) >= 2 and line[0] == '"' and line[-1] == '"':
                line = line[1:-1]
            parts = line.split(",")
            if len(parts) <= max(idx_block, idx_hol5, idx_zeit, idx_zs):
                continue
            rows.append(
                {
                    "Blocknummer": parts[idx_block],
                    "HoL5": parts[idx_hol5],
                    "Zeit": parts[idx_zeit],
                    "Zeitschritt": parts[idx_zs],
                }
            )

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    dt = pd.to_datetime(df["Zeit"].str.replace("_", " ", regex=False), errors="coerce")
    df["dt"] = dt
    df = df.dropna(subset=["dt"]).copy()
    df["HoL5"] = pd.to_numeric(df["HoL5"], errors="coerce")
    df["Blocknummer"] = pd.to_numeric(df["Blocknummer"], errors="coerce")
    df["Zeitschritt"] = pd.to_numeric(df["Zeitschritt"], errors="coerce")
    df = df.dropna(subset=["Blocknummer", "Zeitschritt", "HoL5"]).copy()
    df["Blocknummer"] = df["Blocknummer"].astype(int)
    df["Zeitschritt"] = df["Zeitschritt"].astype(int)

    return df


def _load_germany_union():
    """Return a shapely geometry (EPSG:4326) for Germany, or None if unavailable."""
    if Polygon is None:
        return None

    def _dissolve(gdf):
        geoms = [x for x in gdf.geometry if x is not None and not x.is_empty]
        if not geoms:
            return None
        try:
            from shapely import unary_union as _uu
        except ImportError:
            from shapely.ops import unary_union as _uu

        return _uu(geoms)

    if GERMANY_SHP is not None and GERMANY_SHP.exists():
        if not _HAS_GPD:
            raise RuntimeError("geopandas is required to read GERMANY_SHP; install geopandas or unset GERMANY_SHP.")
        g = _gpd.read_file(GERMANY_SHP)
        if g.crs is not None and str(g.crs) != "EPSG:4326":
            g = g.to_crs(4326)
        return _dissolve(g)

    if not _HAS_GPD:
        return None

    ne_path = None
    if hasattr(_gpd.datasets, "get_path"):
        try:
            ne_path = _gpd.datasets.get_path("naturalearth_lowres")
        except Exception:
            ne_path = None
    if ne_path is None:
        return None

    world = _gpd.read_file(ne_path)
    name_col = "name" if "name" in world.columns else ("NAME" if "NAME" in world.columns else None)
    if name_col is None:
        return None
    names = world[name_col].astype(str).str.strip().str.lower()
    sub = world.loc[names.eq("germany")]
    if sub.empty:
        sub = world.loc[world[name_col].astype(str).str.contains("Germany", case=False, na=False)]
    if sub.empty:
        return None
    if sub.crs is not None and str(sub.crs) != "EPSG:4326":
        sub = sub.to_crs(4326)
    return _dissolve(sub)


@lru_cache(maxsize=512)
def _load_block_mat(mat_path: str) -> dict | None:
    p = Path(mat_path)
    if not p.is_file():
        return None
    with p.open("rb") as f:
        return pickle.load(f)


def _trapkoord_to_polygon(trap: np.ndarray, lon1d: np.ndarray, lat1d: np.ndarray):
    """Build lon/lat polygon from Trapkoord[t, :, :].

    `Trapkoord` stores vertices as (x,y) grid indices. Here we map indices →
    lon/lat using the NCEP coordinate vectors and return a shapely polygon.
    """
    if Polygon is None or trap is None:
        return None
    trap = np.asarray(trap, dtype=float)
    if trap.ndim != 2 or trap.shape[1] < 2:
        return None
    lon1d = np.asarray(lon1d, dtype=float).reshape(-1)
    lat1d = np.asarray(lat1d, dtype=float).reshape(-1)
    verts = []
    for k in range(trap.shape[0]):
        xf, yf = float(trap[k, 0]), float(trap[k, 1])
        if not (np.isfinite(xf) and np.isfinite(yf)):
            return None
        ci = int(np.round(xf))
        rj = int(np.round(yf))
        if ci < 0 or rj < 0 or ci >= lon1d.size or rj >= lat1d.size:
            return None
        lon = float(lon1d[ci])
        lat = float(lat1d[rj])
        # normalize longitudes to [-180, 180] for consistent geometry ops
        lon = ((lon + 180.0) % 360.0) - 180.0
        verts.append((lon, lat))
    if len(verts) < 3:
        return None
    poly = Polygon(verts)
    if not poly.is_valid:
        poly = poly.buffer(0)
    return poly


_de_trap_hit_cache: dict[tuple[str, int, int, int], bool] = {}


def _event_hits_germany(
    year_dir: Path, year: int, block: int, zs: int, germany, *, prepared=None
) -> bool:
    """True if the blocking trapezoid at this timestep intersects Germany.

    Each row in the blocking CSV points to a (block id, timestep). The trapezoid
    vertices are stored in the corresponding `{block}_{year}_python.mat` file.
    We convert that trapezoid to a lon/lat polygon and test intersection with
    the Germany outline.
    """
    if germany is None:
        return True
    key = (str(year_dir.resolve()), year, block, zs)
    if key in _de_trap_hit_cache:
        return _de_trap_hit_cache[key]

    # Load trapezoid vertices for this blocking event & timestep
    mat_path = year_dir / f"{block}_{year}_python.mat"
    data = _load_block_mat(str(mat_path.resolve()))
    if not data or "Trapkoord" not in data:
        _de_trap_hit_cache[key] = False
        return False
    trap_all = np.asarray(data["Trapkoord"])
    if trap_all.ndim != 3 or zs < 0 or zs >= trap_all.shape[0]:
        _de_trap_hit_cache[key] = False
        return False

    lon1d = np.asarray(data["lonncep"]).squeeze()
    lat1d = np.asarray(data["latncep"]).squeeze()
    poly = _trapkoord_to_polygon(trap_all[zs, :, :], lon1d, lat1d)
    if poly is None or poly.is_empty:
        _de_trap_hit_cache[key] = False
        return False

    # Intersection test (use prepared geometry for speed when available)
    if prepared is not None:
        hit = bool(prepared.intersects(poly))
    elif _shapely_prep is not None:
        hit = bool(_shapely_prep(germany).intersects(poly))
    else:
        hit = bool(germany.intersects(poly))
    _de_trap_hit_cache[key] = hit
    return hit


def _blocking_days_by_year(blocking_root: Path, *, germany=None) -> pd.DataFrame:
    """Yearly Aug–Sep blocking-day counts from step-3 CSV.

    If ``germany`` is a shapely geometry (WGS84), only timesteps whose step-2
    trapezoid (``{block}_{year}_python.mat`` → ``Trapkoord``) intersects Germany
    contribute to HoL / Omega / any counts.
    """
    files = sorted(blocking_root.glob(blocking_glob))
    if not files:
        raise FileNotFoundError(f"No blocking files found under {blocking_root} with glob {blocking_glob!r}")

    rows = []
    for fp in files:
        df = _read_blocking_csv(fp)
        if df.empty:
            continue

        year_dir = fp.parent
        try:
            year_csv = int(year_dir.name)
        except ValueError:
            continue

        if germany is not None:
            # Event-level filter: keep only those CSV rows whose trapezoid
            # intersects Germany. This is where we decide which blocking events
            # "remain" for the subsequent daily/yearly aggregation.
            prep = _shapely_prep(germany) if _shapely_prep is not None else None
            keep = []
            for _, r in df.iterrows():
                keep.append(
                    _event_hits_germany(
                        year_dir,
                        year_csv,
                        int(r["Blocknummer"]),
                        int(r["Zeitschritt"]),
                        germany,
                        prepared=prep,
                    )
                )
            df = df.loc[keep].copy()
            if df.empty:
                continue

        df["date"] = df["dt"].dt.floor("D")

        daily = df.groupby("date").agg(
            hol_day=("HoL5", lambda x: bool(np.any(x == 1))),
            omega_day=("HoL5", lambda x: bool(np.any(x == 0))),
        )

        daily = daily.loc[daily.index.month.isin(MONTHS)]
        if daily.empty:
            continue

        by_year = daily.groupby(daily.index.year).agg(
            hol_days=("hol_day", "sum"),
            omega_days=("omega_day", "sum"),
        )
        by_year["any_block_days"] = (daily["hol_day"] | daily["omega_day"]).groupby(daily.index.year).sum()

        rows.append(by_year)

    if not rows:
        raise ValueError("Blocking files were found, but none contained usable timestamps in Aug–Sep.")

    out = pd.concat(rows, axis=0)
    out = out.groupby(out.index).sum().sort_index()
    out.index.name = "year"
    return out


def _regime_days_by_year(labels_da: xr.DataArray, regimes: np.ndarray) -> pd.DataFrame:
    # labels_da: (valid_time,) int regime id
    tdim = labels_da.dims[0]
    t = pd.to_datetime(labels_da[tdim].values)
    labels = pd.Series(labels_da.values.astype(int), index=t, name="regime")

    labels = labels.loc[labels.index.month.isin(MONTHS)]
    if labels.empty:
        raise ValueError("No regime labels found in requested months (Aug–Sep).")

    # Count days per year per regime
    # (If labels are sub-daily, this counts time steps; adjust here if needed.)
    counts = (
        labels.groupby([labels.index.year, labels.values])
        .size()
        .unstack(fill_value=0)
        .sort_index()
    )

    # Ensure all regimes are present as columns
    for r in regimes:
        r = int(r)
        if r not in counts.columns:
            counts[r] = 0

    counts = counts[sorted(counts.columns)]
    counts.index.name = "year"
    counts.columns = [f"regime_{int(c)}_days" for c in counts.columns]
    return counts


# ------------------------
# Compute yearly series + covariance
# ------------------------

_de_trap_hit_cache.clear()
_load_block_mat.cache_clear()

# `germany_union` is created in the previous cell (from the VG5000 shapefile).
# If it is None, the Germany-trapezoid filter is skipped.
if germany_union is None:
    print("Germany trapezoid filter: disabled (no Germany polygon loaded).")
else:
    print("Germany trapezoid filter: using intersection of Trapkoord with Germany outline.")

regime_year = _regime_days_by_year(labels_da, regimes)
metrics = ["any_block_days", "omega_days", "hol_days"]


def _stats_2x2_panel(blocking_year: pd.DataFrame, panel_title: str) -> None:
    """2×2 diagnostic panel comparing regimes vs blocking.

    Uses yearly Aug–Sep counts and shows (regime × metric) heatmaps for:
    - Pearson r
    - Spearman ρ
    - covariance
    Plus a small text panel summarizing the strongest links by |stat|.
    """
    joined = regime_year.join(blocking_year, how="inner")
    print(
        f"\n=== {panel_title} ===\nYears used:",
        int(joined.shape[0]),
        f"({joined.index.min()}–{joined.index.max()})",
    )

    reg_cols = [c for c in joined.columns if c.startswith("regime_") and c.endswith("_days")]

    # Matrices are (regime × metric)
    pearson = pd.DataFrame(index=reg_cols, columns=metrics, dtype=float)
    spearman = pd.DataFrame(index=reg_cols, columns=metrics, dtype=float)
    covar = pd.DataFrame(index=reg_cols, columns=metrics, dtype=float)

    for rc in reg_cols:
        for m in metrics:
            x = joined[rc]
            y = joined[m]
            pearson.loc[rc, m] = float(x.corr(y, method="pearson"))
            spearman.loc[rc, m] = float(x.corr(y, method="spearman"))
            covar.loc[rc, m] = float(x.cov(y))

    # Regime labels for nicer axes
    def _pretty_index(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        out.index = [idx.replace("regime_", "").replace("_days", "") for idx in out.index]
        try:
            out = out.reindex(index=sorted(out.index, key=lambda s: int(s)))
        except Exception:
            pass
        return out

    pearson_h = _pretty_index(pearson)
    spearman_h = _pretty_index(spearman)
    covar_h = _pretty_index(covar)

    # 2×2 panel
    fig, axs = plt.subplots(2, 2, figsize=(10.5, 7.5), constrained_layout=True)

    def _draw_heat(ax, data: pd.DataFrame, *, title: str, vmin: float, vmax: float, cbar_label: str, fmt: str):
        im = ax.imshow(data.values, vmin=vmin, vmax=vmax, cmap="RdBu_r", aspect="auto")
        ax.set_title(title)
        ax.set_xticks(np.arange(data.shape[1]))
        ax.set_xticklabels(["any", "omega", "HoL"], rotation=0)
        ax.set_yticks(np.arange(data.shape[0]))
        ax.set_yticklabels([f"Regime {r}" for r in data.index])
        for i in range(data.shape[0]):
            for j in range(data.shape[1]):
                val = data.values[i, j]
                if np.isfinite(val):
                    ax.text(j, i, format(val, fmt), ha="center", va="center", fontsize=8, color="0.15")
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
        cbar.set_label(cbar_label)

    _draw_heat(
        axs[0, 0],
        pearson_h,
        title=f"{panel_title}: Pearson r",
        vmin=-1,
        vmax=1,
        cbar_label="Pearson r (yearly Aug–Sep counts)",
        fmt="+.2f",
    )

    _draw_heat(
        axs[0, 1],
        spearman_h,
        title=f"{panel_title}: Spearman ρ",
        vmin=-1,
        vmax=1,
        cbar_label="Spearman ρ (yearly Aug–Sep counts)",
        fmt="+.2f",
    )

    vmax_cov = float(np.nanmax(np.abs(covar_h.values))) if np.isfinite(covar_h.values).any() else 1.0
    _draw_heat(
        axs[1, 0],
        covar_h,
        title=f"{panel_title}: Covariance",
        vmin=-vmax_cov,
        vmax=vmax_cov,
        cbar_label="Covariance (yearly Aug–Sep counts)",
        fmt="+.2g",
    )

    # 4th panel: concise ranking summary
    ax = axs[1, 1]
    ax.axis("off")

    def _best_lines(df: pd.DataFrame, label: str, *, fmt: str) -> list[str]:
        lines = [label]
        for m in metrics:
            s = df[m].astype(float)
            if s.isna().all():
                continue
            best_i = (s.abs()).idxmax()
            best_val = s.loc[best_i]
            regime = str(best_i).replace("regime_", "").replace("_days", "")
            lines.append(f"- {m}: Regime {regime} ({format(best_val, fmt)})")
        return lines

    summary = []
    summary += _best_lines(pearson, "Best |Pearson r| by metric:", fmt="+.3f")
    summary += [""]
    summary += _best_lines(spearman, "Best |Spearman ρ| by metric:", fmt="+.3f")
    summary += [""]
    summary += _best_lines(covar, "Best |cov| by metric:", fmt="+.3g")

    ax.text(0, 1, "\n".join(summary), va="top", ha="left", fontsize=9, family="monospace")

    from pathlib import Path
    outdir = Path("figures")
    outdir.mkdir(parents=True, exist_ok=True)
    safe_title = "".join(ch if (ch.isalnum() or ch in "-_.") else "_" for ch in str(panel_title)).strip("_")
    tag_figure_for_save(fig, prefix=f"{REGION.upper()} | ")
    fig.savefig(outdir / f"EU_step10_11_blocking_stats_{safe_title}{BBOX_FIGTAG}.png", dpi=300, bbox_inches="tight")

    plt.show()


if blocking_root is None:
    print("[blocking diagnostic] SKIP: no blocking_root resolved — nothing to plot.")
else:
    blocking_year_all = _blocking_days_by_year(blocking_root, germany=None)
    _stats_2x2_panel(blocking_year_all, "All blocking (no Germany mask)")

    if germany_union is not None:
        blocking_year_de = _blocking_days_by_year(blocking_root, germany=germany_union)
        _stats_2x2_panel(blocking_year_de, "Trapezoid intersects Germany")
